In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import numpy as np
import matplotlib.pyplot as plt

import pickle
import json
import sys
sys.path.append("../../../")


from src.dataset_manager import DatasetConvertor 
from src.context_free.config import ModelConfig, TrainingConfig, MetaConfig
from src.context_free.preprocessing import PreprocessingDataset
from src.context_free.models import createModel
from src.context_free.training import trainModel
from src.context_free.evaluation import evaluateModel

In [ ]:
rawDatasetFolder = "../../../data/raw"
configPath = "../../config/combined_flows_forward_40.json"
modelFolder = "../../../data/models/context_aware"
trafficDataFolder = "../../../data/processed/dpdr"
verbose = True

In [ ]:
config = json.load(open(configPath))
name = config.get("NAME")
len_window = config.get("LEN_WINDOW")
dim_data =  len(config.get("CONTEXT_IDXS"))

In [ ]:
datasetConverter = DatasetConvertor(rawDatasetFolder, randomFlag=False)
with open(configPath, "r") as f:
    config = json.load(f)

name = config.get("NAME")
len_window = config.get("LEN_WINDOW")
dim_data = len(config.get("CONTEXT_IDXS"))
train_ratio = config.get("TRAIN_RATIO")

# =============== Preprocessing ===============
datasetConverter.addDataUnit(config)
dataUnit = datasetConverter.getDataUnit(name)
max_vals, min_vals = dataUnit.getMaxMinMbnVals()
metaConfig = MetaConfig.initialize(
    dim_data=dim_data, 
    len_window=len_window
)
metaConfig.save(f"{modelFolder}/{name}_metaConfig.json")

trainDataUnit, testDataUnit = dataUnit.split(train_ratio)
dataProcessor = PreprocessingDataset(metaConfig)
trainData = dataProcessor.process(trainDataUnit, dataAugment=True)
testData = dataProcessor.process(testDataUnit, dataAugment=False)


In [ ]:
modelConfig = ModelConfig.load(f"{modelFolder}/{name}_modelConfig.json")
model, device = createModel(modelConfig)
model.load_checkpoint(f"{modelFolder}/{name}.pth")
actual, predicted = evaluateModel(model, testData)

plt.plot(actual[0:100])
plt.plot(predicted[0:100])
plt.show()